# VK-SSL: HuBERT + codec targets (E3 smoke / real DAC)
Runtime -> Change runtime type -> GPU (T4). Запускайте ячейки по порядку.

In [ ]:
!nvidia-smi | head -15
!git clone -b codec-targets https://github.com/vladislavdauer/VK-SSL.git 2>&1 | tail -1 || true
# если ветка не запушена: загрузите codec_targets.patch в /content и выполните ячейку ниже

In [ ]:
%cd /content
!git clone -b preparing_backbone https://github.com/vladislavdauer/VK-SSL.git VK-SSL-patched
%cd VK-SSL-patched
# загрузите codec_targets.patch слева (Files) в /content, затем:
!git apply --whitespace=nowarn /content/codec_targets.patch && git status --short

In [ ]:
!pip -q install pytorch_lightning torchmetrics sentencepiece descript-audio-codec
!pip -q install -e .

In [ ]:
# unit-тесты (stub-кодек)
!python -m unittest experiments.hubert_train.test.test_codec_targets -v 2>&1 | tail -12

In [ ]:
# данные: dev-clean ~337 МБ
!python experiments/ctc_train/download_dev_subset_libri.py

In [ ]:
# E3 smoke: tiny + stub (проверка трубопровода)
!PYTHONPATH=. python -m experiments.hubert_train.train_hubert --librispeech-path ./librispeech --model-size tiny --targets codec-online --codec stub --codec-num-q 3 --label-rate 50 --sanity_check --max-steps 50 --gpus 1 --precision 32-true

In [ ]:
# E3 real: DAC-16k, Q=4, веса по уровням, base на dev-clean (только проверка, не обучение для статьи)
!PYTHONPATH=. python -m experiments.hubert_train.train_hubert --librispeech-path ./librispeech --model-size tiny --targets codec-online --codec dac-16khz --codec-num-q 4 --codebook-weights 1,0.5,0.25,0.125 --label-rate 50 --sanity_check --max-steps 100 --gpus 1 --precision bf16-mixed